# Databricks/PySpark for Data Analysts/Engineers — Module 9: Optimization and Performance

Throughout the course we've mentioned Catalyst, execution plans, and shuffle costs --
this module pulls it all together: how Spark optimizes queries, when caching data is
worth it, how to control the number of partitions, and how to read `.explain()`
seriously.

## Table of contents
1. [Catalyst and Tungsten: how Spark optimizes queries](#sec1)
2. [cache() and persist()](#sec2)
3. [In-memory partitioning: repartition vs. coalesce](#sec3)
4. [Reading execution plans in depth](#sec4)
5. [Summary and exercises](#sec5)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

# Extra JVM options (--add-opens) are needed when running Spark locally on Java 17+ and
# using pandas UDFs / Apache Arrow (Module 8) -- newer JDKs block, by default, the
# reflective off-heap memory access that Arrow needs. In Databricks you don't need to
# set this by hand -- the platform handles it for you.
_jvm_options = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("spark_course")
    .config("spark.driver.extraJavaOptions", _jvm_options)
    .config("spark.executor.extraJavaOptions", _jvm_options)
    .getOrCreate()
)
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. Catalyst and Tungsten: how Spark optimizes queries

We've mentioned them since Module 1 -- time to pull it together. When you build a chain
of transformations, Spark does NOT execute it step by step exactly as you wrote it.
Instead:

1. **Logical plan** -- Spark translates your code (DataFrame API or SQL) into an
   abstract tree of operations, regardless of how you wrote it.
2. **Optimization by Catalyst** -- optimization rules transform this plan: combining
   successive filters into one, *predicate pushdown* (pushing filters as close to the
   data source as possible), *column pruning* (reading only the needed columns), choosing
   a join strategy (broadcast vs. shuffle, Module 4), and many others.
3. **Physical plan** -- the optimized logical plan turns into concrete operations to run
   on the cluster.
4. **Tungsten** -- a lower-level execution engine that manages memory and generates JVM
   bytecode directly for the physical plan (*whole-stage code generation*), instead of
   interpreting it step by step.

> ⚡ **Why SQL and the DataFrame API give identical performance**
>
> Whether you write `df.filter(...).select(...)` or `SELECT ... WHERE ...` (Module 5), BOTH land on the EXACT SAME logical plan, go through the SAME Catalyst rules, and end up as the SAME physical plan. Choosing a style is about code readability, never performance -- we already mentioned this in Module 5.

In [ ]:
from pyspark.sql import functions as F

# Two separate filters written apart -- Catalyst combines them into one step anyway
plan = orders.filter(F.col("status") == "Delivered").filter(
    F.col("customer_id") > 100
).select("order_id", "customer_id")
plan.explain()


<a id="sec2"></a>
## 2. cache() and persist()

Because Spark uses lazy evaluation (Module 1), EVERY time you call an action on a
DataFrame, the whole chain of transformations gets computed FROM SCRATCH -- including
re-reading the source data. If you use the same (expensive-to-compute) DataFrame
MULTIPLE times, `cache()`/`persist()` remembers the result after the first computation.

In [ ]:
orders_with_value = orders.join(order_items, "order_id").groupBy(
    "order_id", "customer_id"
).agg(F.sum(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount"))).alias("value"))

orders_with_value.cache()

# First action -- actually computes the whole chain AND stores the result in memory
print(f"Row count: {orders_with_value.count()}")

# Further actions on the same DataFrame -- read from cache, no recomputation
orders_with_value.filter(F.col("value") > 1000).show(5)


> ⚠️ **cache() without a first action does nothing**
>
> `.cache()` (like any transformation) is LAZY -- by itself it just MARKS a DataFrame as 'to be remembered'. The actual computation and storing in memory only happens at the FIRST action called AFTER `cache()`. Forgetting this is a common mistake: someone calls `cache()`, then wonders why the next run is still slow, because they never actually triggered an action to make the cache get populated in the first place.

In [ ]:
# persist() with an explicit level -- more control than cache() (which is just MEMORY_AND_DISK)
from pyspark import StorageLevel

orders_with_value.unpersist()
orders_with_value.persist(StorageLevel.MEMORY_ONLY)
orders_with_value.count()
print(orders_with_value.storageLevel)


> 🧱 **When caching is WORTH it, and when it's a waste of memory**
>
> Caching makes sense when a DataFrame is used MULTIPLE times in downstream steps (e.g. several different aggregations on the same filtered dataset) -- you save repeating expensive computation. Caching a DataFrame used only ONCE is pure waste of cluster memory. Also remember to call `.unpersist()` once a DataFrame is no longer needed -- otherwise it keeps occupying memory unnecessarily for the rest of the session.

<a id="sec3"></a>
## 3. In-memory partitioning: repartition vs. coalesce

The number of partitions a DataFrame has in memory determines how many parallel tasks
Spark can run at once. `repartition(n)` reshuffles data into EXACTLY `n` partitions
(always does a shuffle, can both increase and decrease the partition count).
`coalesce(n)` DECREASES the partition count WITHOUT a full shuffle (merges neighboring
partitions locally) -- faster, but only works one way (decreasing).

In [ ]:
print(f"Partition count before: {orders.rdd.getNumPartitions()}")

orders_more_partitions = orders.repartition(8)
print(f"After repartition(8): {orders_more_partitions.rdd.getNumPartitions()}")

orders_fewer_partitions = orders_more_partitions.coalesce(2)
print(f"After coalesce(2): {orders_fewer_partitions.rdd.getNumPartitions()}")


In [ ]:
# repartition by a column -- useful before expensive grouping/joining operations on
# that same column, so rows with the same key land in the same partition
orders_by_customer = orders.repartition(4, "customer_id")
print(f"Partition count: {orders_by_customer.rdd.getNumPartitions()}")


> ⚠️ **Too few partitions AND too many are both a problem**
>
> TOO FEW partitions (e.g. 1-2 on a cluster with 20 executors) means most executors sit idle -- work isn't spread out evenly. TOO MANY tiny partitions means the overhead of managing thousands of tiny tasks outweighs the real gain from parallelism. A good starting point in practice: 2-4 partitions per CPU core available in the cluster -- but this always requires experimenting on your actual data.

<a id="sec4"></a>
## 4. Reading execution plans in depth

`.explain()` by default shows only the physical plan. The `mode="formatted"` parameter
(or `True` for all plans) shows MORE detail -- including the logical plan before and
after Catalyst's optimization.

In [ ]:
query = (
    orders.join(customers, "customer_id")
    .filter(F.col("segment") == "Premium")
    .groupBy("city")
    .agg(F.count("*").alias("count"))
)
query.explain(mode="formatted")


Things worth looking for in the printed plan:
- `Filter` -- did the filter land as close to the data read as possible (predicate pushdown)?
- `Project` -- is Spark reading only the needed columns (column pruning)?
- `BroadcastHashJoin` / `SortMergeJoin` -- which join strategy was chosen (Module 4)?
- `Exchange` -- marks a SHUFFLE at that point in the plan -- every occurrence is potential network cost.

In [ ]:
query.explain(mode="cost")


> 🧱 **The Spark UI instead of reading text**
>
> In practice, especially in Databricks, the same information is much more comfortable to read in the graphical Spark UI (the SQL / DataFrame tab) than in text-based `.explain()` -- it also shows the EXECUTION TIME of every stage, not just the plan itself. Text-based `.explain()` is still irreplaceable, though, when working locally or when you want to check a plan quickly right in your code/notebook.

<a id="sec5"></a>
## 5. Summary and exercises

In this module we covered:
- Catalyst and Tungsten -- how Spark optimizes DataFrame API/SQL code down to one shared physical plan,
- `cache()`/`persist()` -- remembering a result for repeatedly used DataFrames, and that they're lazy,
- `repartition()` vs. `coalesce()` -- controlling the number of in-memory partitions,
- reading `.explain(mode=...)` in more depth: `Filter`, `Project`, join type, `Exchange` (shuffle).

### 📝 Exercise 1: Caching a repeatedly used DataFrame

Create a filtered DataFrame `orders.filter(status == 'Cancelled')`, call `cache()` on it, count it with `.count()`, then run TWO different aggregations on the same cached DataFrame.

<details>
<summary>Show solution</summary>

```python
cancelled = orders.filter(F.col("status") == "Cancelled").cache()
print(f"Count: {cancelled.count()}")

cancelled.groupBy("customer_id").count().orderBy(F.col("count").desc()).show(5)
cancelled.groupBy("employee_id").count().orderBy(F.col("count").desc()).show(5)
```

</details>

### 📝 Exercise 2: repartition vs. coalesce

Check the partition count of `products`, increase it to 6 via `repartition`, then decrease it back to 2 via `coalesce` -- compare the partition counts at each step.

<details>
<summary>Show solution</summary>

```python
print(f"Initially: {products.rdd.getNumPartitions()}")
more = products.repartition(6)
print(f"After repartition(6): {more.rdd.getNumPartitions()}")
fewer = more.coalesce(2)
print(f"After coalesce(2): {fewer.rdd.getNumPartitions()}")
```

</details>

### 📝 Exercise 3: Finding Exchange in the plan

Join `orders` with `customers` (without a broadcast hint) and call `.explain(mode="formatted")` -- find the `Exchange` occurrence in the output (a shuffle).

<details>
<summary>Show solution</summary>

```python
joined = orders.join(customers, "customer_id")
joined.explain(mode="formatted")
# Look for "Exchange" nodes in the output -- they mark a shuffle on both sides of the join
```

</details>

### 📝 Exercise 4: Broadcast vs. no broadcast -- comparing plans

Run the SAME join `orders` with `employees` twice: once without a hint, once with `F.broadcast(employees)`. Compare `.explain()` for both -- do they differ in the number of `Exchange` nodes?

<details>
<summary>Show solution</summary>

```python
without_broadcast = orders.join(employees, "employee_id")
with_broadcast = orders.join(F.broadcast(employees), "employee_id")

print("=== Without broadcast ===")
without_broadcast.explain()
print("=== With broadcast ===")
with_broadcast.explain()
```

</details>

> 🔥 **Challenge: caching in a full multi-step pipeline**
>
> Build a pipeline: (1) join `orders`, `customers`, `order_items`, compute `item_value` -- PRESERVE (`cache()`) the result of this step; (2) on the cached result compute THREE different reports: total value per city, total value per segment, and the top 5 customers by total value. Measure approximate execution time with `time.time()` before and after -- with caching and (for comparison) without it.

<details>
<summary>Show solution</summary>

```python
import time

def build_joined():
    return (
        orders.join(customers, "customer_id")
        .join(order_items, "order_id")
        .withColumn(
            "item_value",
            F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")),
        )
    )

# Without cache -- each of the three actions recomputes the whole chain from scratch
start = time.time()
no_cache = build_joined()
no_cache.groupBy("city").agg(F.sum("item_value")).collect()
no_cache.groupBy("segment").agg(F.sum("item_value")).collect()
no_cache.groupBy("customer_id").agg(F.sum("item_value").alias("total")).orderBy(
    F.col("total").desc()
).limit(5).collect()
print(f"Without cache: {time.time() - start:.2f}s")

# With cache -- the chain is computed once, then three actions read from memory
start = time.time()
with_cache = build_joined().cache()
with_cache.count()  # forces the first computation and stores it in the cache
with_cache.groupBy("city").agg(F.sum("item_value")).collect()
with_cache.groupBy("segment").agg(F.sum("item_value")).collect()
with_cache.groupBy("customer_id").agg(F.sum("item_value").alias("total")).orderBy(
    F.col("total").desc()
).limit(5).collect()
print(f"With cache: {time.time() - start:.2f}s")
```

On a dataset as small as this course's, the difference may be minor or even invisible (the overhead of caching itself can be comparable to the cost of recomputing) -- the real savings show up on large, distributed data, where re-reading and reprocessing source data is genuinely expensive.

</details>

## What's next?

That was the last purely technical PySpark module. Starting in **Module 10** we move
on to Delta Lake and the lakehouse architecture -- the foundations of a real
Databricks environment.